# AC Performance

**BQ2:** How can seasonal weather changes, air-con performance and humidity levels be analysed to detect zones that are at risk of air-con degradation or mould growth?

Compares July-August 2018 with 2019 for Floors 3-7, zones 1, 2, 4 and 5, with and without filled readings. It produces `outputs/bq2-zone-screen/inspection_shortlist.csv`, which the risk matrix (`bq2_risk_matrix.ipynb`) uses for the **AC energy up** and **faulty AC meter** labels.

## Methods

| Step | Rule |
|---|---|
| Inputs | Existing raw/cleaned minutes and cached weather; no new filling or partial-day scaling. Observed-only readings must be present and unchanged by cleaning. |
| Energy | Inferred working days with all 1,440 AC-plus-plug minutes; every assigned AC meter required. |
| Environment | Continuous 60-minute startup windows; complete steady-on hours after 120 AC-on minutes. Thresholds are frozen from raw-observed 2018. |
| Reference | Fit/select using 2018 only: temperature-plus-plug versus enthalpy-plus-plug, expanding temporal validation and mean/activity benchmarks. |
| Comparability | At least 20 reference days/four weeks; at least 10 evaluation days/four weeks inside the 2018 predictor convex hull, by session where relevant. |
| Uncertainty | Equal total day weights; 2,000 calendar-week bootstrap refits with fixed specification/cohort. Nominal conditional 95% intervals, without multiplicity adjustment. |
| Interpretation | Stronger leads require reference MAE ≤ mean-only MAE, ≥90% valid bootstrap replicates and an adverse interval in both input versions. Weak references remain labelled; mean-only sensitivity uses the same supported rows. |

These are project screening rules. Working-day/plug labels are activity proxies; 2018 is not known healthy. Missing results do not establish healthy equipment. Clock alignment, calibration, controls and loads remain uncertain. Full tables are saved as CSVs; intermediate displays are previews.

In [10]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Find the project folder: the one holding both data-cleaning/ and mould-ac-degradation/
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (p / "data-cleaning").is_dir() and (p / "analytics/mould-ac-degradation/bq2_utils.ipynb").is_file())
analysis_dir = repo_root / "analytics/mould-ac-degradation"
sys.path.insert(0, str(analysis_dir / "src"))
helper_notebook = analysis_dir / "bq2_utils.ipynb"
get_ipython().run_line_magic("run", f'-i "{helper_notebook}"')

from bq2 import zone_screen as screen
sys.path.insert(0, str(repo_root / "analytics/baseline"))
from humidity_risk_baseline import load_bangkok_era5_weather
output_dir = Path(analysis_dir / "outputs/bq2-zone-screen")
output_dir.mkdir(parents=True, exist_ok=True)
weather = load_bangkok_era5_weather(repo_root / "baseline/data/bangkok_era5_hourly_2018-2019.csv")
helpers = {name: globals()[name] for name in ["add_outdoor_psychrometrics", "humidity_ratio_gkg", "ac_on_threshold"]}
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 8)

## 1. Check inputs

Audit timestamps, channel validity, coverage, filling and hourly reconciliation before fitting. Floor 6's empty malformed tail row is excluded locally; trailing minutes remain missing. Floor 7 uses its explicit day-first source format.

In [11]:
%%capture --no-display --no-stderr
evidence = screen.audit_and_features(repo_root, weather, helpers, output_dir)
quality = evidence["zone_quality"]
display(quality[["zone_id", "ac_meters", "full_clean_joint_coverage_pct",
                 "with_fills_2018_joint_coverage_pct", "with_fills_2019_joint_coverage_pct",
                 "suspect_sensor", "meter_mapping_review"]].round(2))
display(evidence["timestamp_audit"].query("invalid_timestamps > 0"))
assert evidence["channel_quality"].quality_log_missing_agrees.all()
assert evidence["hourly_reconciliation"][["availability_mismatches", "value_mismatches"]].sum().sum() == 0
print("All 95 relevant channel missingness records and 80 hourly comparisons reconcile.")
print("Longest requested-window raw plateaus:")
display(evidence["sensor_checks"].groupby("stream").raw_scope_max_constant_minutes.max())


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\panhc\\OneDrive\\Documents\\GitHub\\SBA-Final-Project\\data-cleaning\\outputs\\analytics\\zone_hourly.csv'

F4-Z1 and F7-Z1 have near-zero AC channels requiring meter/configuration checks. F5-Z1 environmental readings are held for sensor verification; energy is assessed separately. These flags do not diagnose failed equipment or verify sensor placement.

## 2. Fit eligible comparisons

Keep unavailable metrics and their reasons. Energy may remain usable during environmental outages. Reference models that miss the simple benchmark retain a validation caution.

In [6]:
%%capture --no-display --no-stderr
result = screen.run_screen(evidence, output_dir, repetitions=2000)
summary = result["model_summary"]
display(summary.groupby(["analysis", "variant"]).agg(
    fitted=("reference_cv_MAE", "count"), supported=("reportable", "sum"),
    passes_reference_validation=("lead_eligible", "sum")))
cv = result["cross_validation"]
assert cv.date.dt.year.eq(2018).all()
assert pd.to_datetime(cv.train_last_date).lt(cv.date).all()
assert result["shared_cohort_checks"].max_abs_input_or_target_difference.dropna().lt(1e-8).all()
assert not summary.query("analysis == 'cooling'").reportable.any()
print("No cooling comparison meets the coverage/common-support rules.")


NameError: name 'evidence' is not defined

## 3. Compare references and inspection leads

The mean-only sensitivity uses each model's exact supported cohort. It tests dependence on the weather/session reference, not a weather-adjusted efficiency change. The saved `inspection_shortlist.csv` lists every zone and its status.

In [7]:
benchmark = screen.benchmark_sensitivity(evidence, result, output_dir, repetitions=2000)
shortlist = screen.inspection_shortlist(quality, summary, benchmark)
shortlist.to_csv(output_dir / "inspection_shortlist.csv", index=False)
display(shortlist[["zone_id", "inspection_category", "validated_metrics_in_both_versions", "validation_passes_only_one_version"]])
focus = ["F3-Z4", "F5-Z4", "F7-Z2"]
display(summary[summary.zone_id.isin(focus) & summary.analysis.isin(["energy", "moisture"])][[
    "zone_id", "analysis", "variant", "supported_days", "mean_residual", "residual_pct",
    "bootstrap_low", "bootstrap_high", "reference_cv_MAE", "mean_only_cv_MAE", "lead_eligible"]].round(3))
display(benchmark[benchmark.zone_id.isin(focus)].round(3))


NameError: name 'evidence' is not defined

### Findings

- **F3-Z4:** energy departure +13.3%/+15.2% with/without fills, with a reference that improves on the mean benchmark. Inspect operating conditions; lower recorded moisture and unavailable cooling comparisons do not establish degradation.
- **F5-Z4:** +36.7% in both versions, but reference validation is weak. The same-cohort mean sensitivity is +33.0%, supporting an electrical/operating investigation.
- **F7-Z2:** recorded moisture +1.15/+1.14 g/kg on 33/29 days, with an observed-only validation caution. Sensor, ventilation, moisture-load and control explanations remain open.
- F3-Z1/F3-Z2 show lower supported energy. Other zones retain QA, provisional or unavailable statuses; none is certified healthy.